# 🐾 Dog vs. Cat Image Classification Using a 3-Layer Convolutional Neural Network (CNN)
**Academic Project Report & Implementation**

This notebook implements an end-to-end binary image classifier using a Convolutional Neural Network with **strictly three Conv2D layers**, trained from scratch without transfer learning.

In [ ]:
# 1. Imports and Reproducibility Setup
import os
import shutil
import random
import zipfile
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import tensorflow as tf
from tensorflow.keras import layers, models

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow Version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
# 2. Dataset Ingestion, Validation & De-Corruption
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32

# Download dataset if running in fresh Colab session
DATA_ZIP = "cats_and_dogs_filtered.zip"
if not os.path.exists("data/cats") and not os.path.exists("archive/PetImages"):
    url = "https://download.mlcc.google.com/mledu-datasets/cats_and_dogs_filtered.zip"
    tf.keras.utils.get_file(DATA_ZIP, origin=url, extract=True, cache_dir=".")

def validate_image_file(path):
    try:
        if os.path.getsize(path) == 0:
            return False
        with Image.open(path) as img:
            img.verify()
        with Image.open(path) as img:
            img.convert("RGB")
        raw = tf.io.read_file(path)
        _ = tf.io.decode_image(raw, channels=3, expand_animations=False)
        return True
    except Exception:
        return False

print("Validation helper ready.")

In [ ]:
# 3. Define the Strict 3-Conv2D Layer CNN Model
def build_model(input_shape=(128, 128, 3), lr=0.001):
    model = models.Sequential([
        layers.Input(shape=input_shape, name="input_image"),
        # Layer 1
        layers.Conv2D(32, (3, 3), activation="relu", padding="same", name="conv2d_1"),
        layers.MaxPooling2D((2, 2), name="maxpool_1"),
        # Layer 2
        layers.Conv2D(64, (3, 3), activation="relu", padding="same", name="conv2d_2"),
        layers.MaxPooling2D((2, 2), name="maxpool_2"),
        # Layer 3
        layers.Conv2D(128, (3, 3), activation="relu", padding="same", name="conv2d_3"),
        layers.MaxPooling2D((2, 2), name="maxpool_3"),
        # Dense Classification Head
        layers.Flatten(name="flatten"),
        layers.Dense(128, activation="relu", name="dense_128"),
        layers.Dropout(0.5, name="dropout_regularization"),
        layers.Dense(1, activation="sigmoid", name="output_prediction")
    ], name="Dog_Cat_3Layer_CNN")
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=["accuracy"]
    )
    return model

model = build_model()
model.summary()

In [ ]:
# 4. Single-Image Inference Demonstration
def predict_dog_or_cat(image_path, model):
    pil_img = Image.open(image_path).convert("RGB")
    resized = pil_img.resize((128, 128), Image.Resampling.BILINEAR)
    arr = np.array(resized, dtype=np.float32) / 255.0
    tensor = np.expand_dims(arr, axis=0)
    sig_val = float(model.predict(tensor, verbose=0)[0][0])
    pred_class = "Dog" if sig_val >= 0.5 else "Cat"
    conf = sig_val * 100 if pred_class == "Dog" else (1.0 - sig_val) * 100
    
    plt.figure(figsize=(4, 4))
    plt.imshow(pil_img)
    plt.title(f"Predicted: {pred_class} ({conf:.1f}%)
Sigmoid: {sig_val:.4f}", fontweight="bold")
    plt.axis("off")
    plt.show()
    return pred_class, conf

print("Prediction function defined.")